<a href="https://colab.research.google.com/github/jeffheaton/app_deep_learning/blob/main/assignments/assignment_yourname_t81_558_class4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# T81-558: Applications of Deep Neural Networks
* Instructor: [Jeff Heaton](https://sites.washu.edu/jeffheaton/), McKelvey School of Engineering, [Washington University in St. Louis](https://engineering.washu.edu/index.html)
* For more information visit the [class website](https://sites.washu.edu/jeffheaton/t81-558/).

**Module 4 Assignment: Classification and Regression Neural Network**

**Student Name: Your Name**

# Assignment Instructions

For this assignment, you will use the **crx.csv** dataset.  This dataset is publicly available and can be found [here](https://archive.ics.uci.edu/ml/datasets/credit+approval). You should use the CSV file on my data site, at this location: [crx.csv](https://data.heatonresearch.com/data/t81-558/crx.csv) because it includes column headers.  The primary use for this dataset is binary classification. There are 15 attributes, plus a target column that contains only + or -.  Some columns contain missing values.

You should train a neural network and return the predictions.  You will submit these predictions to the **submit** function.  See [Assignment #1](https://github.com/jeffheaton/app_deep_learning/blob/main/assignments/assignment_yourname_t81_558_class1.ipynb) for details on how to submit an assignment or check that one was submitted.

Complete the following tasks:

* Your task is to replace missing values in columns *a2* and *a14* with values estimated by a neural network (one neural network for *a2* and another for *a14*).
* Your submission file will contain the same headers as the source CSV: *a1*, *a2*, *s3*, *a4*, *a5*, *a6*, *a7*, *a8*, *a9*, *a10*, *a11*, *a12*, *a13*, *a14*, *a15*, and *a16*.
* You should only need to modify *a2* and *a14*.
* Neural networks can be much more powerful at filling missing variables than median and mean.
* Train two neural networks to predict *a2* and *a14*.  
* The *y* (target) for training the two nets will be *a2* and *a14*, depending on which you are trying to fill.
* The x for training the two nets will be 's3','a8','a9','a10','a11','a12','a13','a15'.  These are chosen because it is important not to use any columns with missing values; also, it could cause unwanted bias if we include the ultimate target (*a16*).
* ONLY predict new values for missing values in *a2* and *a14*.
* You will likely get this small warning:  Warning: The mean of column a14 differs from the solution file by 0.20238937709643778. (might not matter if small)



# Get key


In [1]:
import os
from dotenv import load_dotenv

load_dotenv()

api_key = os.getenv("T81_558_KEY")

# Assignment Submission Library

Run the following code to install the assignment submission package (`jh_submit`). All of the submission, listing, and file-checking utilities used in this notebook live in this package.

In [2]:
# Assignment submission package
import jh_submit

# Assignment #4 Sample Code

The following code provides a starting point for this assignment.

In [20]:
import copy
import torch

# Make use of a GPU or MPS (Apple) if one is available.  (see module 3.2)
import torch
has_mps = torch.backends.mps.is_built()
device = "mps" if has_mps else "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")


# Early stopping (see module 3.4)
class EarlyStopping:
    def __init__(self, patience=5, min_delta=0, restore_best_weights=True):
        self.patience = patience
        self.min_delta = min_delta
        self.restore_best_weights = restore_best_weights
        self.best_model = None
        self.best_loss = None
        self.counter = 0
        self.status = ""

    def __call__(self, model, val_loss):
        if self.best_loss is None:
            self.best_loss = val_loss
            self.best_model = copy.deepcopy(model.state_dict())
        elif self.best_loss - val_loss >= self.min_delta:
            self.best_model = copy.deepcopy(model.state_dict())
            self.best_loss = val_loss
            self.counter = 0
            self.status = f"Improvement found, counter reset to {self.counter}"
        else:
            self.counter += 1
            self.status = f"No improvement in the last {self.counter} epochs"
            if self.counter >= self.patience:
                self.status = f"Early stopping triggered after {self.counter} epochs."
                if self.restore_best_weights:
                    model.load_state_dict(self.best_model)
                return True
        return False

Using device: cpu


In [24]:
import pandas as pd
from scipy.stats import zscore
from sklearn.model_selection import train_test_split

# Begin assignment
df = pd.read_csv("https://data.heatonresearch.com/data/t81-558/crx.csv",na_values=['?'])



from sklearn.preprocessing import StandardScaler

def prepare_data(df, X_names, y_name):
    filt_df = df[df[y_name].notna()].copy()
    X = filt_df[X_names].copy()
    y = filt_df[y_name].copy()
    numeric_cols = X.select_dtypes(include="number").columns
    X = pd.get_dummies(X, dtype=float)
    scaler = StandardScaler()
    X[numeric_cols] = scaler.fit_transform(X[numeric_cols])
    x = torch.tensor(
        X.values,
        dtype=torch.float32
    )
    y = torch.tensor(
        y.values,
        dtype=torch.float32
    ).reshape(-1, 1)
    return x, y, X.columns, numeric_cols, scaler
def create_model(n_features):
    return nn.Sequential(
        nn.Linear(n_features, 20),
        nn.ReLU(),
        nn.Linear(20, 10),
        nn.ReLU(),
        nn.Linear(10, 1)
    ).to(device)
def train_model(
    model,
    x_train,
    y_train,
    x_val=None,
    y_val=None,
    epochs=500,
    batch_size=32,
    lr=0.001
):
    train_dataset = TensorDataset(x_train, y_train)
    train_loader = DataLoader(
        train_dataset,
        batch_size=batch_size,
        shuffle=True
    )
    optimizer = optim.Adam(model.parameters(), lr=lr)
    loss_fn = nn.MSELoss()
    es = EarlyStopping()
    for epoch in range(epochs):
        # Training
        model.train()
        for x_batch, y_batch in train_loader:
            x_batch = x_batch.to(device)
            y_batch = y_batch.to(device)
            optimizer.zero_grad()
            output = model(x_batch)
            loss = loss_fn(output, y_batch)
            loss.backward()
            optimizer.step()
        # Validation + early stopping
        if x_val is not None:
            model.eval()
            with torch.no_grad():
                val_output = model(x_val.to(device))
                val_loss = loss_fn(
                    val_output,
                    y_val.to(device)
                )
            if es(model, val_loss):
                break
    return model, epoch + 1
def predict_with_kfold(seed, k, df, X_names, y_name):
    x, y, columns, numeric_cols, scaler = prepare_data(
        df,
        X_names,
        y_name
    )

    kf = KFold(
        n_splits=k,
        shuffle=True,
        random_state=seed
    )

    fold_scores = []
    fold_epochs = []

    # ----- Cross-validation -----
    for fold, (train_idx, val_idx) in enumerate(kf.split(x), 1):

        x_train, x_val = x[train_idx], x[val_idx]
        y_train, y_val = y[train_idx], y[val_idx]

        model = create_model(x.shape[1])

        model, n_epochs = train_model(
            model,
            x_train,
            y_train,
            x_val,
            y_val
        )

        model.eval()

        with torch.no_grad():
            pred = model(x_val.to(device)).cpu()

        rmse = torch.sqrt(
            nn.MSELoss()(pred, y_val)
        ).item()

        fold_scores.append(rmse)
        fold_epochs.append(n_epochs)

        print(
            f"Fold {fold}: "
            f"RMSE={rmse:.4f}, "
            f"epochs={n_epochs}"
        )

    # ----- CV results -----
    print(
        f"\nMean RMSE: {np.mean(fold_scores):.4f} "
        f"+/- {np.std(fold_scores):.4f}"
    )

    final_epochs = round(np.mean(fold_epochs))

    print(f"Final training epochs: {final_epochs}")

    # ----- Train final model on ALL known values -----
    final_model = create_model(x.shape[1])

    final_model, _ = train_model(
        final_model,
        x,
        y,
        epochs=final_epochs
    )

    return final_model, columns, numeric_cols, scaler

In [25]:
X_names = [
    's3', 'a8', 'a9', 'a10',
    'a11', 'a12', 'a13', 'a15'
]
def predict_missing (cur_df, X_names, target_col):
    df = cur_df.copy()
    cur_model, cur_X_columns, cur_num_cols, cur_scaler = predict_with_kfold(
        seed=42,
        k=5,
        df=df,
        X_names=X_names,
        y_name=target_col
    )
    # Predict missing a2
    missing_mask = df[target_col].isna()
    X_missing = df.loc[missing_mask, X_names].copy()
    X_missing = pd.get_dummies(X_missing, dtype=float)
    X_missing = X_missing.reindex(
        columns=cur_X_columns,
        fill_value=0
    )
    X_missing[cur_num_cols] = cur_scaler.transform(
        X_missing[cur_num_cols]
    )
    x_missing = torch.tensor(
        X_missing.values,
        dtype=torch.float32
    )

    cur_model.eval()
    with torch.no_grad():
        cur_pred = (
            cur_model(x_missing.to(device))
            .cpu()
            .numpy()
            .flatten()
        )
    df.loc[missing_mask, target_col] = cur_pred
    return df
df_submit = predict_missing( cur_df = df, X_names = X_names, target_col= "a2"  )
df_submit = predict_missing( cur_df = df_submit, X_names = X_names, target_col= "a14"  )

Fold 1: RMSE=11.1077, epochs=43
Fold 2: RMSE=11.3791, epochs=46
Fold 3: RMSE=9.7821, epochs=58
Fold 4: RMSE=12.3810, epochs=22
Fold 5: RMSE=10.8888, epochs=34

Mean RMSE: 11.1077 +/- 0.8367
Final training epochs: 41
Fold 1: RMSE=237.5826, epochs=77
Fold 2: RMSE=138.6405, epochs=34
Fold 3: RMSE=131.7918, epochs=52
Fold 4: RMSE=134.1438, epochs=36
Fold 5: RMSE=178.8220, epochs=129

Mean RMSE: 164.1962 +/- 40.5111
Final training epochs: 66


In [23]:
df

,a1,a2,s3,a4,a5,a6,a7,a8,a9,a10,a11,a12,a13,a14,a15,a16
0,b,30.83,0.000,u,g,w,v,1.25,t,t,1,f,g,202.0,0,+
1,a,58.67,4.460,u,g,q,h,3.04,t,t,6,f,g,43.0,560,+
2,a,24.50,0.500,u,g,q,h,1.50,t,f,0,f,g,280.0,824,+
3,b,27.83,1.540,u,g,w,v,3.75,t,t,5,t,g,100.0,3,+
4,b,20.17,5.625,u,g,w,v,1.71,t,f,0,f,s,120.0,0,+
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
685,b,21.08,10.085,y,p,e,h,1.25,f,f,0,f,g,260.0,0,-
686,a,22.67,0.750,u,g,c,v,2.00,f,t,2,t,g,200.0,394,-
687,a,25.25,13.500,y,p,ff,ff,2.00,f,t,1,t,g,200.0,1,-
688,b,17.92,0.205,u,g,aa,v,0.04,f,f,0,f,g,280.0,750,-


0      30.83
1      58.67
2      24.50
3      27.83
4      20.17
       ...  
685    21.08
686    22.67
687    25.25
688    17.92
689    35.00
Name: a2, Length: 690, dtype: float64

In [27]:
file="/home/clever/Projects/Python/ml_course/app_deep_learning/assignments/assignment_Maksim_t81_558_class4.ipynb"
## Submit assignment
jh_submit.client.submit(source_file=file,data=[df_submit],key=api_key,course="t81-558",no=4)

Success: Submitted Assignment 4 (t81-558) for maksim:
This is your first submission of this assignment.

Note: The mean difference 0.011934486548657475 for column 'a2' is acceptable and is less than the maximum allowed value of '1.0' for this assignment.
Note: The mean difference 0.5034686434096329 for column 'a14' is acceptable and is less than the maximum allowed value of '1.0' for this assignment.
No warnings or errors (only notes), you will probably do well, but no guarantee. :-)
